# Tokenization

স্ক্র্যাচ থেকে তৈরি একটি Byte-Pair Encoding (BPE) tokenizer:

1. একটি toy corpus থেকে merge-গুলো শেখা,
2. সেই merge-গুলো দিয়ে text encode/decode করা,
3. এবং দেখানো যে BPE একটি আগে-কখনো-না-দেখা word-কেও সুন্দরভাবে সামলায় — যা word-level tokenization গঠনগতভাবেই করতে পারে না।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। প্রথম cell-গুলো শুধু function সংজ্ঞায়িত করে; পুরো demo-টি শেষ cell-এর `main()` চালায়।

In [ ]:
from collections import Counter

END_OF_WORD = "</w>"   # word boundary চিহ্নিত করে, যাতে কোনো merge কখনো দুটি word পার না হয়

CORPUS = [
    "low", "lower", "lowest", "low", "lower",
    "new", "newer", "newest", "new",
    "wide", "wider", "widest",
]

## 1. BPE training: সবচেয়ে ঘন ঘন আসা symbol pair বারবার merge করা

প্রতিটি word-কে প্রথমে আলাদা character-এ ভাঙা হয় (শেষে `</w>` যোগ করে)। তারপর প্রতি ধাপে সব পাশাপাশি symbol pair গোনা হয় — word-টি corpus-এ কতবার এসেছে সেই অনুযায়ী weighted — এবং সবচেয়ে বেশি আসা pair-টিকে একটি নতুন symbol-এ merge করা হয়। Merge-গুলোর **ক্রম** সংরক্ষণ করা হয়, কারণ নতুন text encode করার সময় ঠিক সেই ক্রমেই এগুলো প্রয়োগ করতে হয়।

In [ ]:
def word_to_symbols(word):
    """'low' -> ['l', 'o', 'w', '</w>']"""
    return list(word) + [END_OF_WORD]


def get_pair_counts(word_freqs):
    """প্রতিটি পাশাপাশি symbol pair গোনে, word-টি কতবার এসেছে সেই অনুযায়ী weighted।"""
    pair_counts = Counter()
    for symbols, freq in word_freqs:
        for i in range(len(symbols) - 1):
            pair_counts[(symbols[i], symbols[i + 1])] += freq
    return pair_counts


def merge_pair(pair, word_freqs):
    """`pair`-এর প্রতিটি occurrence-কে একটিমাত্র merged symbol দিয়ে প্রতিস্থাপন করে।"""
    merged_symbol = "".join(pair)
    new_word_freqs = []
    for symbols, freq in word_freqs:
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_word_freqs.append((new_symbols, freq))
    return new_word_freqs


def train_bpe(corpus, num_merges):
    word_counts = Counter(corpus)
    word_freqs = [(word_to_symbols(word), freq) for word, freq in word_counts.items()]

    merges = []  # merge-গুলোর ক্রমানুসারী list; নতুন text encode করতে এই ক্রম গুরুত্বপূর্ণ
    for step in range(num_merges):
        pair_counts = get_pair_counts(word_freqs)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=pair_counts.get)
        merges.append(best_pair)
        word_freqs = merge_pair(best_pair, word_freqs)
        print(f"  merge {step + 1:2d}: {best_pair} -> {''.join(best_pair)!r} "
              f"(appeared {pair_counts[best_pair]} times)")

    vocab = sorted({sym for symbols, _ in word_freqs for sym in symbols})
    return merges, vocab

## 2. Encoding এবং word-level tokenizer-এর সাথে তুলনা

`bpe_encode` একটি নতুন word-কে character-এ ভেঙে শেখা merge-গুলো **শেখার ক্রমেই** প্রয়োগ করে। তুলনার জন্য `word_level_lookup` হলো একটি word-level tokenizer: word-টি হুবহু vocabulary-তে থাকলে সেটিই token, না থাকলে `<unk>`।

In [ ]:
def bpe_encode(word, merges):
    """শেখা merge-গুলো, শেখার ক্রমেই (IN LEARNED ORDER), একটি নতুন word-এ প্রয়োগ করে।"""
    symbols = word_to_symbols(word)
    for pair in merges:
        i = 0
        new_symbols = []
        merged_symbol = "".join(pair)
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


def word_level_lookup(word, known_vocab):
    """একটি word-level tokenizer: হয় হুবহু word-টি জানা, নয়তো এটি <unk>।"""
    return [word] if word in known_vocab else ["<unk>"]

## 3. পুরো demo: training, দেখা word encode করা, এবং একটি অদেখা word

`main()` প্রথমে corpus-এর word frequency দেখায়, 12টি BPE merge শেখে এবং চূড়ান্ত vocabulary print করে। তারপর training-এ দেখা কয়েকটি word encode করে, এবং শেষে `"widening"` — যা training-এ কখনো পুরোটা দেখা যায়নি — BPE ও word-level দুই tokenizer দিয়েই encode করে। BPE ছোট কিন্তু অর্থবহ জানা টুকরোয় (সবচেয়ে খারাপ ক্ষেত্রে একক character পর্যন্ত) fallback করে; word-level tokenizer-এর এমন কোনো fallback নেই — word-টি সোজা `<unk>`।

In [ ]:
def main():
    print("Training corpus (word frequencies):")
    print(f"  {dict(Counter(CORPUS))}\n")

    print("Learning BPE merges:")
    merges, vocab = train_bpe(CORPUS, num_merges=12)

    print(f"\nFinal learned vocabulary ({len(vocab)} symbols): {vocab}")

    print("\n" + "=" * 70)
    print("ENCODING SEEN WORDS")
    print("=" * 70)
    for word in ["low", "newest", "wider"]:
        tokens = bpe_encode(word, merges)
        print(f"  {word!r:10s} -> {tokens}")

    print("\n" + "=" * 70)
    print("ENCODING A WORD NEVER SEEN DURING TRAINING")
    print("=" * 70)
    unseen_word = "widening"  # "widen"->"wide"+"..." গঠন share করে, কিন্তু পুরো word-টি কখনো দেখা যায়নি
    bpe_tokens = bpe_encode(unseen_word, merges)
    word_level_tokens = word_level_lookup(unseen_word, set(CORPUS))
    print(f"  BPE tokenizer:        {unseen_word!r} -> {bpe_tokens}")
    print(f"  word-level tokenizer: {unseen_word!r} -> {word_level_tokens}")
    print("\n  -> BPE falls back to smaller, still-meaningful known pieces (down to")
    print("     individual characters in the worst case) instead of giving up.")
    print("     A word-level tokenizer has no such fallback -- it's simply unknown.")


main()